In [1]:
# !pip3 install mosaicml-streaming

In [2]:
import torch
import torch.nn as nn
import pandas as pd
from datasets import Audio
from transformers import AutoTokenizer, AddedToken
from streaming import MDSWriter
from streaming.base.format.mds.encodings import Encoding, _encodings
from streaming import LocalDataset
import streaming
import numpy as np
from tqdm import tqdm
from glob import glob
import os
import json
from multiprocess import Pool
import itertools

def block_diagonal_concat_inverted(*masks, dtype=torch.bfloat16):
    total_size = sum(mask.size(0) for mask in masks)
    combined_mask = torch.zeros(total_size, total_size, dtype=dtype)

    current_pos = 0

    for mask in masks:
        size = mask.size(0)
        combined_mask[current_pos:current_pos + size, current_pos:current_pos + size] = mask
        current_pos += size

    min_value = torch.finfo(dtype).min if dtype.is_floating_point else torch.iinfo(dtype).min
    inverted_mask = torch.where(combined_mask == 1, torch.tensor(0, dtype=dtype), min_value)
    return inverted_mask.unsqueeze(0)

def chunks(l, n):
    for i in range(0, len(l), n):
        yield (l[i: i + n], i // n)

def multiprocessing(strings, function, cores=6, returned=True):
    df_split = chunks(strings, len(strings) // cores)
    pool = Pool(cores)
    pooled = pool.map(function, df_split)
    pool.close()
    pool.join()

    if returned:
        return list(itertools.chain(*pooled))

class UInt32(Encoding):
    def encode(self, obj) -> bytes:
        return obj.tobytes()

    def decode(self, data: bytes):
        return np.frombuffer(data, np.uint32)

_encodings['uint32'] = UInt32

columns = {
    'input_ids': 'uint32',
    'position_ids': 'uint32',
    'attention_mask': 'uint32',
    'audio': 'str',
    'text': 'str'
}
hashes = 'sha1', 'xxh64'

/home/ubuntu/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [3]:
tokenizer = AutoTokenizer.from_pretrained('Qwen/Qwen2.5-1.5B')
extra = [
    AddedToken('<|endofspeech|>'), 
    AddedToken('<|whole|>'), 
    AddedToken('<|streaming|>'),
    AddedToken('<|segment|>'),
    AddedToken('<|word|>'),
    AddedToken('<|'),
    AddedToken('|>')
]
for i in range(16384):
    extra.append(AddedToken(f'<|s{i}|>'))
tokenizer.add_tokens(extra)

16391

In [4]:
import gc

def collator(batch, batch_position_ids):
    input_ids = []
    position_ids = []
    masks = []
    for i in range(len(batch)):
        l = len(batch[i])
        input_ids.extend(batch[i])
        position_ids.extend(batch_position_ids[i])
        masks.append(l)
    
    return {
        'input_ids': np.array(input_ids).astype(np.uint32),
        'position_ids': np.array(position_ids).astype(np.uint32),
        'attention_mask': np.array(masks).astype(np.uint32),
        'audio': '',
        'text': '',
    }

def slice_and_balance(nested_list, size):
    first = []
    balance = []
    current_size = 0

    for sublist in nested_list:
        if current_size < size:
            remaining_space = size - current_size
            if len(sublist) <= remaining_space:
                first.append(sublist)
                current_size += len(sublist)
            else:
                first.append(sublist[:remaining_space])
                balance.append(sublist[remaining_space:])
                current_size = size
        else:
            balance.append(sublist)
    
    return first, balance

In [5]:
!rm -rf tokenized-8k-qwen
!mkdir tokenized-8k-qwen

In [6]:
import time

sequence_length = 10240
def loop(files, block_size = sequence_length):
    files, index = files
    out_root = f'tokenized-8k-qwen/tokenized-{index}'
    os.system(f'rm -rf {out_root}')
    count = 0
    temp = []
    position_ids = []
    last_block, last_position_block = None, None
    with MDSWriter(out=out_root, columns=columns, compression=None, hashes=hashes) as out:
        for file in tqdm(files):
            
            try:
                with open(file) as fopen:
                    prompt = json.load(fopen)
            except:
                continue

            if len(prompt) < 5:
                continue
            
            outputs = tokenizer(prompt, add_special_tokens = False)
            position = range(len(outputs['input_ids']))
            length = len(outputs['input_ids'])

            if length > block_size:
                continue
            
            if count + length > block_size:
                o = collator(temp, position_ids)
                out.write(o)
                temp = [outputs['input_ids']]
                position_ids = [position]
                count = length
                
            else:
                temp.append(outputs['input_ids'])
                position_ids.append(range(len(outputs['input_ids'])))
                count += len(outputs['input_ids'])
        
        if len(temp):
            o = collator(temp, position_ids)
            out.write(o)
            

In [7]:
files = glob('*_done/*')
len(files)

35791048

In [8]:
from multiprocess import Pool

chunks = chunks(files, 500000)
pool = Pool(30)
pooled = pool.map(loop, chunks)
pool.close()
pool.join()

100%|██████████| 500000/500000 [25:28<00:00, 327.12it/s]


In [9]:
# multiprocessing(files, loop, 50, returned = False)

In [10]:
folders = sorted(glob('tokenized-8k-qwen/tokenized-*'), key = lambda x: int(x.split('-')[-1]))
folders

['tokenized-8k-qwen/tokenized-0',
 'tokenized-8k-qwen/tokenized-1',
 'tokenized-8k-qwen/tokenized-2',
 'tokenized-8k-qwen/tokenized-3',
 'tokenized-8k-qwen/tokenized-4',
 'tokenized-8k-qwen/tokenized-5',
 'tokenized-8k-qwen/tokenized-6',
 'tokenized-8k-qwen/tokenized-7',
 'tokenized-8k-qwen/tokenized-8',
 'tokenized-8k-qwen/tokenized-9',
 'tokenized-8k-qwen/tokenized-10',
 'tokenized-8k-qwen/tokenized-11',
 'tokenized-8k-qwen/tokenized-12',
 'tokenized-8k-qwen/tokenized-13',
 'tokenized-8k-qwen/tokenized-14',
 'tokenized-8k-qwen/tokenized-15',
 'tokenized-8k-qwen/tokenized-16',
 'tokenized-8k-qwen/tokenized-17',
 'tokenized-8k-qwen/tokenized-18',
 'tokenized-8k-qwen/tokenized-19',
 'tokenized-8k-qwen/tokenized-20',
 'tokenized-8k-qwen/tokenized-21',
 'tokenized-8k-qwen/tokenized-22',
 'tokenized-8k-qwen/tokenized-23',
 'tokenized-8k-qwen/tokenized-24',
 'tokenized-8k-qwen/tokenized-25',
 'tokenized-8k-qwen/tokenized-26',
 'tokenized-8k-qwen/tokenized-27',
 'tokenized-8k-qwen/tokenized-

In [11]:
!rm -rf packing-qwen2

In [12]:
with MDSWriter(out='packing-qwen2', columns=columns, compression=None, hashes=hashes) as out:
    for f in folders:
        try:
            dataset = LocalDataset(local=f)
            for i in tqdm(range(len(dataset))):
                out.write(dataset[i])
        except Exception as e:
            print(e)
            pass

100%|██████████| 8043/8043 [00:02<00:00, 3890.62it/s]


In [14]:
dataset = LocalDataset('packing-qwen2')
(len(dataset) * 10240) / 1e9

19.3455104